# Lesson 04 - टूल उपयोग डिज़ाइन पैटर्न

इस पाठ में आप Microsoft Agent Framework (Python) का उपयोग करके AI एजेंट्स के लिए **टूल उपयोग** डिज़ाइन पैटर्न सीखेंगे। हम शामिल हैं:

- `@tool` डेकोरेटर और टाइप किए गए पैरामीटर्स के साथ फंक्शन टूल्स को परिभाषित करना
- मॉडल को यह समझाने के लिए टूल स्कीमाएं प्रदान करना कि प्रत्येक टूल क्या करता है
- `approval_mode` के साथ टूल निष्पादन को नियंत्रित करना
- Pydantic मॉडल और `response_format` के माध्यम से **संगठित आउटपुट** लौटाना

परिदृश्य एक **यात्रा बुकिंग एजेंट** का है जो गंतव्यों को खोज सकता है, उपलब्धता की जांच कर सकता है, और उड़ान की जानकारी प्राप्त कर सकता है।


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool डेकोरेटर के साथ उपकरण परिभाषित करना

`@tool` डेकोरेटर एक साधारण पायथन फ़ंक्शन को एक उपकरण में बदल देता है जिसे एजेंट कॉल कर सकता है।  
मुख्य बातें:

- **डॉकस्ट्रिंग** वह उपकरण विवरण बन जाती है जिसे मॉडल देखता है।  
- **टाइप एनोटेशन** (जिसमें विवरण के साथ `Annotated` शामिल है) उपकरण स्कीमा को परिभाषित करते हैं।  
- `approval_mode` नियंत्रित करता है कि क्या उपयोगकर्ता को प्रत्येक कॉल को निष्पादित करने से पहले अनुमोदित करना आवश्यक है।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## एक एजेंट को कई टूल्स के साथ बनाना

सभी तीन टूल्स को क्लाइंट को पास करें ताकि मॉडल उपयोगकर्ता के प्रश्न का उत्तर देने के लिए आवश्यक किसी भी टूल को बुला सके।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## उपकरणों के साथ संरचित आउटपुट

`response_format` को एक Pydantic मॉडल पर सेट करके, एजेंट को मुक्त-प्रकार के पाठ के बजाय एक अच्छी तरह से टाइप किया हुआ JSON ऑब्जेक्ट वापस करने के लिए बाध्य किया जाता है। जब डाउनस्ट्रीम कोड को परिणाम को प्रोग्रामेटिकली उपभोग करने की आवश्यकता होती है तो यह उपयोगी होता है।


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## टूल अनुमोदन पैटर्न

`@tool` पर `approval_mode` पैरामीटर नियंत्रित करता है कि टूल कॉल को निष्पादित करने से पहले मानव अनुमोदन की आवश्यकता है या नहीं:

| Mode | व्यवहार |
|---|---|
| `"never_require"` | टूल स्वचालित रूप से चलता है — उपयोगकर्ता की पुष्टि की आवश्यकता नहीं है। |
| `"always_require"` | हर कॉल को निष्पादित होने से पहले उपयोगकर्ता द्वारा अनुमोदित किया जाना चाहिए। |

ऐसे टूल्स के लिए `"always_require"` का उपयोग करें जिनके साइड-इफेक्ट्स होते हैं (जैसे उड़ान बुक करना, क्रेडिट कार्ड से चार्ज करना) ताकि मानव नियंत्रण में बना रहे।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## सारांश

इस पाठ में आपने सीखा कि कैसे:

1. **उपकरण निर्धारित करें** `@tool` डेकोरेटर का उपयोग करके टाइप किए गए पैरामीटर और डॉकस्ट्रिंग्स के साथ जो उपकरण स्कीमा के रूप में कार्य करते हैं।
2. **कई उपकरणों को संयोजित करें** ताकि एजेंट उन्हें क्रम में कॉल कर सके और जटिल प्रश्नों का उत्तर दे सके।
3. **संरचित आउटपुट लौटाएं** पायडांटिक मॉडल को `response_format` के रूप में पास करके।
4. **उपकरण अनुमोदन को नियंत्रित करें** `approval_mode` के साथ ताकि संवेदनशील कार्यों के लिए मानव शामिल रहे।

ये पैटर्न विश्वसनीय, उत्पादन-तैयार एजेंट बनाने की नींव बनाते हैं जो बाहरी प्रणालियों के साथ सुरक्षित रूप से इंटरैक्ट कर सकते हैं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यह दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके अनुवादित किया गया है। जबकि हम शुद्धता के लिए प्रयासरत हैं, कृपया ध्यान रखें कि स्वचालित अनुवाद में त्रुटियां या असामान्यताएं हो सकती हैं। मूल भाषा में मूल दस्तावेज़ को अधिकृत स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफ़हमी या गलत व्याख्या के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
